# Step 8 — Potential and flow network

**Learning objective:** Plot equipotential lines and streamlines using exported OGS vector data.

**Run from top to bottom in VS Code.** Each notebook executes one workshop step and shows its output. Select the working OGS Postprocessing Python kernel.

**Prerequisites:** run earlier numbered steps. `GMSH2OGS` and `ogs` must be on PATH when used; Gmsh is imported from the selected Python kernel in Step 2. **Every image below represents a distinct quantity.** Green PASS messages verify the stated condition only.

In [ ]:
from pathlib import Path
import sys, subprocess
import numpy as np
import matplotlib.pyplot as plt
import meshio
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents, Path.cwd().parent) if (p / "models" / "excavation.prj").exists()), None)
assert ROOT is not None, "Open the extracted project directory in VS Code before running this notebook"
print('Training root:', ROOT)
print('Python:', sys.executable)
def run(*command):
    command = [str(c) for c in command]
    print('RUN:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=ROOT, check=True)


**Explain:** the OGS output array `pressure` is model potential, and `v` is the corresponding model flux vector. The white wall is impermeable by excluded geometry. The figures below use the actual VTU files.

In [ ]:
import matplotlib.tri as mtri
from matplotlib.patches import Rectangle
from scipy.interpolate import griddata
import vtk
from vtk.util.numpy_support import vtk_to_numpy
import vtuIO
from ogs6py import ogs
RESULTS = ROOT/'results'
PLOTS=RESULTS/'notebook_plots';PLOTS.mkdir(exist_ok=True,parents=True)
files60=sorted(RESULTS.glob('excavation_diffusion_ts_1_*.vtu'))
files50=sorted((RESULTS/'bc50').glob('excavation_diffusion_ts_1_*.vtu'))
assert files60 and files50, 'Run Steps 6 and 7 first'
PATH60,PATH50=files60[-1],files50[-1]
def load_ogs_vtu(path):
    rd=vtk.vtkXMLUnstructuredGridReader();rd.SetFileName(str(path));rd.Update();mesh=rd.GetOutput()
    xy=vtk_to_numpy(mesh.GetPoints().GetData())[:,:2]
    cells=np.array([[mesh.GetCell(i).GetPointId(j) for j in range(3)] for i in range(mesh.GetNumberOfCells())],dtype=int)
    u=vtk_to_numpy(mesh.GetPointData().GetArray('pressure'))
    v_arr=mesh.GetPointData().GetArray('v')
    v=vtk_to_numpy(v_arr) if v_arr is not None else None
    return xy,cells,u,v
reader=vtuIO.VTUIO(str(PATH60),dim=2)
print('VTUinterface loaded:',PATH60.name)

**Run and observe:** filled potential contours (common scale), thin dark equipotential lines, and blue streamlines where `v` is available. Streamlines come from a numerically interpolated vector field; interpolation is masked outside valid finite elements.

In [ ]:
def plot_flow_net(ax, vtu_path, title):
    xy, cells, potential, velocity = load_ogs_vtu(vtu_path)
    tri = mtri.Triangulation(xy[:, 0], xy[:, 1], cells)
    contour = ax.tricontourf(tri, potential, levels=np.linspace(30, 60, 31), cmap='coolwarm')
    ax.tricontour(tri, potential, levels=np.arange(30, 60.1, 2.5),
                  colors='#202833', alpha=0.6, linewidths=0.55)

    if velocity is not None:
        xs = np.linspace(0, 65, 230)
        ys = np.linspace(0, 30, 120)
        X, Y = np.meshgrid(xs, ys)
        VX = griddata(xy, velocity[:, 0], (X, Y), method='linear')
        VY = griddata(xy, velocity[:, 1], (X, Y), method='linear')
        # Interpolation across the slit is not valid; mask outside actual FEM triangles.
        inside = tri.get_trifinder()(X, Y) >= 0
        valid = inside & np.isfinite(VX) & np.isfinite(VY)
        VX = np.ma.masked_where(~valid, VX)
        VY = np.ma.masked_where(~valid, VY)
        ax.streamplot(xs, ys, VX, VY, density=1.3,
                      color='#075a9e', linewidth=0.65, arrowsize=0.7,
                      minlength=0.18, maxlength=5)
    else:
        print(f'No point vector v in {vtu_path.name}; plotting equipotentials only')

    # Thin impermeable wall in the benchmark geometry (0.2 m wide).
    wall_x = 32.5
    ax.add_patch(Rectangle((wall_x - 0.1, 8), 0.2, 22,
                           facecolor='white', edgecolor='#192a3b', linewidth=1.2, zorder=9))
    ax.scatter([wall_x], [8], color='#192a3b', s=12, zorder=10)
    ax.text(wall_x + 1.0, 23, 'impermeable wall', rotation=90,
            fontsize=9, va='center', color='#192a3b')
    ax.set(xlim=(0, 65), ylim=(0, 30), xlabel='x [m]', ylabel='y [m]', title=title)
    ax.set_aspect('equal')
    ax.grid(False)
    return contour

fig, axes = plt.subplots(2, 1, figsize=(12, 10), layout='constrained')
cf = plot_flow_net(axes[0], PATH60, 'Reference: potential left 60 / right 30')
plot_flow_net(axes[1], PATH50, 'Exercise: potential left 50 / right 30')
cbar = fig.colorbar(cf, ax=axes, orientation='horizontal', shrink=0.70, pad=0.07, aspect=38)
cbar.set_label('Potential [model units] — not kPa')
fig.suptitle('Excavation benchmark: equipotentials and flux streamlines', fontsize=15)
output_path = PLOTS / 'flow_network_both_cases.png'
fig.savefig(output_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', output_path)

**Interpretation:** Filled colours = scalar potential (`pressure`, model units). Thin dark curves = equipotential contours. Blue arrows/curves = **streamlines of the exported `v` field**; directions follow that array. White = wall void (excluded region). Streamlines pass around the wall tip. This output is not a calibrated hydrogeological seepage plot in m/s. **Next:** independently check discrete FEM balance.